# NMR match simulator — PySpice/ngspice

Interactive Binder notebook for the two-capacitor matching circuit. Move the sliders and the plots update after release. Run the notebook first to see the interactive plots. The notebook uses PySpice to simulate the circuit and ngspice as the backend simulator.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

import PySpice.Logging.Logging as Logging
from PySpice.Spice.Netlist import Circuit
from PySpice.Unit import *

Logging.setup_logging()

Z0 = 50.0
f_start, f_stop = 10e6, 800e6
n_points = 6001
f_marker = 300e6
target_Icoil = 1.0

Cin0, Ctune0, Cgnd0 = 3.2, 33.6, 3.2
L0, Rs0 = 8.0, 0.1


def branch_current(analysis, name):
    name = name.lower()
    for key in analysis.branches.keys():
        if str(key).lower() == name:
            return np.array(analysis.branches[key], dtype=complex)
    raise RuntimeError(f"Branch current '{name}' not found.")


def db20(x):
    return 20 * np.log10(np.clip(np.abs(x), 1e-12, None))


def simulate(Cin_pF, Ctune_pF, Cgnd_pF, L_nH, Rs_ohm):
    """
    port -- Cin -- n1 ---- Ctune ---- n2 -- Cgnd -- ground
                   |                 |
                   +---- L -- Rs ----+

    Vtest = 1 Vrms AC source.
    Vsense = 0 V source used only to measure coil current.
    """
    c = Circuit("NMR match")
    c.raw_spice += """
Vtest port 0 DC 0 AC 1
Vsense coil_mid coil_loss DC 0
"""
    c.C("in", "port", "n1", Cin_pF @ u_pF)
    c.C("tune", "n1", "n2", Ctune_pF @ u_pF)
    c.C("gnd", "n2", c.gnd, Cgnd_pF @ u_pF)
    c.L("coil", "n1", "coil_mid", L_nH @ u_nH)
    c.R("coil", "coil_loss", "n2", Rs_ohm @ u_Ohm)

    # DC references only; negligible at RF.
    c.R("n1_leak", "n1", c.gnd, 1e12 @ u_Ohm)
    c.R("n2_leak", "n2", c.gnd, 1e12 @ u_Ohm)

    a = c.simulator(temperature=25, nominal_temperature=25).ac(
        start_frequency=f_start @ u_Hz,
        stop_frequency=f_stop @ u_Hz,
        number_of_points=n_points,
        variation="lin",
    )

    f = np.array(a.frequency, dtype=float)
    Vport = np.array(a["port"], dtype=complex)
    Iin = -branch_current(a, "vtest")
    Icoil = branch_current(a, "vsense")

    Zin = Vport / Iin
    S11 = (Zin - Z0) / (Zin + Z0)
    Pfwd = np.abs((Vport / (1 + S11)))**2 / Z0
    return f, Zin, S11, Icoil, Pfwd


def marker_text(f, Zin, S11, Icoil, Pfwd):
    k = np.argmin(np.abs(f - f_marker))
    eta = abs(Icoil[k]) / np.sqrt(max(Pfwd[k], 1e-12))
    P_needed = (target_Icoil / eta)**2
    text = (
        f"{f[k]/1e6:.1f} MHz | "
        f"Zin = {Zin[k].real:.2f} + j{Zin[k].imag:.2f} Ω | "
        f"S11 = {db20(S11[k]):.2f} dB\n"
        f"Icoil/√Pfwd = {eta:.3g} A/√W | "
        f"Pfwd for {target_Icoil:.1f} A = {P_needed:.3g} W"
    )
    return k, text


def plot_probe(Cin, Ctune, Cgnd, L, Rs):
    f, Zin, S11, Icoil, Pfwd = simulate(Cin, Ctune, Cgnd, L, Rs)
    k, text = marker_text(f, Zin, S11, Icoil, Pfwd)

    fig, (ax_mag, ax_smith) = plt.subplots(
        2, 1, figsize=(8, 8), gridspec_kw={"height_ratios": [1, 1.5]}
    )

    ax_mag.plot(f / 1e6, db20(S11))
    ax_mag.plot(f[k] / 1e6, db20(S11[k]), "o")
    ax_mag.axvline(f_marker / 1e6, linestyle="--", alpha=0.5)
    ax_mag.set_xlim(f_start / 1e6, f_stop / 1e6)
    ax_mag.set_xlabel("Frequency / MHz")
    ax_mag.set_ylabel("|S11| / dB")
    ax_mag.grid(True, alpha=0.3)

    theta = np.linspace(0, 2*np.pi, 500)
    ax_smith.plot(np.cos(theta), np.sin(theta), "k", alpha=0.25)
    ax_smith.plot(0, 0, "ko")
    ax_smith.text(0.03, 0.03, "50 Ω")
    ax_smith.plot(S11.real, S11.imag)
    ax_smith.plot(S11[k].real, S11[k].imag, "o")
    ax_smith.set_aspect("equal", adjustable="box")
    ax_smith.set_xlim(-1.05, 1.05)
    ax_smith.set_ylim(-1.05, 1.05)
    ax_smith.set_xlabel("Re(S11)")
    ax_smith.set_ylabel("Im(S11)")
    ax_smith.grid(True, alpha=0.3)

    fig.suptitle(text)
    plt.show()


ui = widgets.VBox([
    widgets.FloatSlider(value=Cin0, min=0.1, max=30, step=0.1, description="Cin / pF", continuous_update=False),
    widgets.FloatSlider(value=Ctune0, min=1, max=200, step=0.1, description="Ctune / pF", continuous_update=False),
    widgets.FloatSlider(value=Cgnd0, min=0.1, max=30, step=0.1, description="Cgnd / pF", continuous_update=False),
    widgets.FloatSlider(value=L0, min=1, max=50, step=0.1, description="L / nH", continuous_update=False),
    widgets.FloatSlider(value=Rs0, min=0.01, max=2, step=0.01, description="Rs / Ω", continuous_update=False),
])

out = widgets.interactive_output(
    plot_probe,
    {"Cin": ui.children[0], "Ctune": ui.children[1], "Cgnd": ui.children[2], "L": ui.children[3], "Rs": ui.children[4]},
)

display(ui, out)